# v9_b Step 5b: Compile steroid metabolites and reaction steps

We started with four feature tables, each matched to multiple sets of predicted reactions.

- Collate feature annotations per feature table. Substrate specific rnx match takes precedent over others. 
- Redo tally on unmatched. Analyze manually.
- Merge cells and supernatant data.

In [ ]:
import os
import json
import pandas as pd
from asari.tools import match_features as mf

input_dir = 'results/step5a_f2net/'
outdir = 'results/step5b_combined_net_anno/'     
os.makedirs(outdir, exist_ok=True)

In [ ]:
def summarize_rxn_steps_per_feature(feature_matched):
    product = feature_matched.copy()
    sequence = [(product['id'], product['formula'], '', ''), ]
    input_cpd = product.get('input_cpd', None)
    while input_cpd:
        rxn = product.get('reaction')
        sequence.append((input_cpd['id'], input_cpd['formula'], rxn['formula_difference'], rxn['reactionIDs']))
        product = input_cpd
        input_cpd = input_cpd.get('input_cpd', None)
        
    sequence.reverse()  # reverse to get the correct order from input to final product
    return sequence

In [ ]:

features = json.load(open(input_dir + 'progesterone_3step_products_supernatant_features.json'))
unmatched = json.load(open(input_dir + 'progesterone_3step_products_supernatant_unmatched.json'))

In [ ]:
summarize_rxn_steps_per_feature(next(f['matched'] for f in features if 'matched' in f))

In [ ]:
steps = [len(summarize_rxn_steps_per_feature(x['matched'])) for x in features if 'matched' in x]
len(features), len(steps), steps.count(4), steps.count(3), steps.count(2), steps.count(1)

In [ ]:
# Do DnCl, including estrone and estradiol
#
# add match_formula, synonyms, bioreactions
#
DnCl_matched_features_pellets, unmatched = {}, []

for feat in json.load(open(input_dir + 'estrone_3step_products_pellets_features.json')):
    if 'matched' in feat:
        _steps = summarize_rxn_steps_per_feature(feat['matched'])
        DnCl_matched_features_pellets[feat['id']] = {
            'id': feat['id'],
            'mz': feat['mz'],
            'rtime': feat['rtime'],
            'source': feat['source'],
            "ion_relation": feat['ion_relation'],
            "neutral_mass": feat['neutral_formula_mass'],   # derivatized
            'bioreactions': list(zip(feat['enzyme'], feat['substrate'])),
            'rxn_steps': [_steps],
            'inferred_formula': _steps[-1][1],      # not derivatized
            'synonyms': [_steps[-1][0]]
        }

for feat in json.load(open(input_dir + 'estradiol_3step_products_pellets_features.json')):
    if 'matched' in feat:
        _steps = summarize_rxn_steps_per_feature(feat['matched'])
        if feat['id'] in DnCl_matched_features_pellets:
            DnCl_matched_features_pellets[feat['id']]['rxn_steps'].append(_steps)
            DnCl_matched_features_pellets[feat['id']]['synonyms'].append(_steps[-1][0])
        else:
            DnCl_matched_features_pellets[feat['id']] = {
                'id': feat['id'],
                'mz': feat['mz'],
                'rtime': feat['rtime'],
                'source': feat['source'],
                "ion_relation": feat['ion_relation'],
                "neutral_mass": feat['neutral_formula_mass'],   # derivatized
                'bioreactions': list(zip(feat['enzyme'], feat['substrate'])),
                'rxn_steps': [_steps],
                'inferred_formula': _steps[-1][1],      # not derivatized
                'synonyms': [_steps[-1][0]]
            }

for feat in json.load(open(input_dir + 'estrone_3step_products_pellets_unmatched.json')
                      ) + json.load(open(input_dir + 'estradiol_3step_products_pellets_unmatched.json')):
    if feat['id'] not in DnCl_matched_features_pellets:
        if 'matched' in feat and feat['id'] not in DnCl_matched_features_pellets:
            _steps = summarize_rxn_steps_per_feature(feat['matched'])
            DnCl_matched_features_pellets[feat['id']] = {
                'id': feat['id'],
                'mz': feat['mz'],
                'rtime': feat['rtime'],
                'source': feat['source'],
                "ion_relation": feat['ion_relation'],
                "neutral_mass": feat['neutral_formula_mass'],   # derivatized
                'bioreactions': list(zip(feat['enzyme'], feat['substrate'])),
                'rxn_steps': [_steps],
                'inferred_formula': _steps[-1][1],      # not derivatized
                'synonyms': [_steps[-1][0]]
            }
        else:
            unmatched.append(feat)

print(len(DnCl_matched_features_pellets), len(unmatched), len(set([f['id'] for f in unmatched])))


In [ ]:

json.dump(list(DnCl_matched_features_pellets.values()), 
          open(os.path.join(outdir, 'DnCl_anno_features_pellets.json'), 'w'), indent=4)
json.dump(unmatched, open(os.path.join(outdir, 'DnCl_unmatched_features_pellets.json'), 'w'), indent=4)

In [ ]:
DnCl_matched_features_supernatant, unmatched = {}, []

for feat in json.load(open(input_dir + 'estrone_3step_products_supernatant_features.json')):
    if 'matched' in feat:
        _steps = summarize_rxn_steps_per_feature(feat['matched'])
        DnCl_matched_features_supernatant[feat['id']] = {
            'id': feat['id'],
            'mz': feat['mz'],
            'rtime': feat['rtime'],
            'source': feat['source'],
            "ion_relation": feat['ion_relation'],
            "neutral_mass": feat['neutral_formula_mass'],   # derivatized
            'bioreactions': list(zip(feat['enzyme'], feat['substrate'])),
            'rxn_steps': [_steps],
            'inferred_formula': _steps[-1][1],      # not derivatized
            'synonyms': [_steps[-1][0]]
        }

for feat in json.load(open(input_dir + 'estradiol_3step_products_supernatant_features.json')):
    if 'matched' in feat:
        _steps = summarize_rxn_steps_per_feature(feat['matched'])
        if feat['id'] in DnCl_matched_features_supernatant:
            DnCl_matched_features_supernatant[feat['id']]['rxn_steps'].append(_steps)
            DnCl_matched_features_supernatant[feat['id']]['synonyms'].append(_steps[-1][0])
        else:
            DnCl_matched_features_supernatant[feat['id']] = {
                'id': feat['id'],
                'mz': feat['mz'],
                'rtime': feat['rtime'],
                'source': feat['source'],
                "ion_relation": feat['ion_relation'],
                "neutral_mass": feat['neutral_formula_mass'],   # derivatized
                'bioreactions': list(zip(feat['enzyme'], feat['substrate'])),
                'rxn_steps': [_steps],
                'inferred_formula': _steps[-1][1],      # not derivatized
                'synonyms': [_steps[-1][0]]
            }

for feat in json.load(open(input_dir + 'estrone_3step_products_supernatant_unmatched.json')
                      ) + json.load(open(input_dir + 'estradiol_3step_products_supernatant_unmatched.json')):
    if feat['id'] not in DnCl_matched_features_supernatant:
        if 'matched' in feat and feat['id'] not in DnCl_matched_features_supernatant:
            _steps = summarize_rxn_steps_per_feature(feat['matched'])
            DnCl_matched_features_supernatant[feat['id']] = {
                'id': feat['id'],
                'mz': feat['mz'],
                'rtime': feat['rtime'],
                'source': feat['source'],
                "ion_relation": feat['ion_relation'],
                "neutral_mass": feat['neutral_formula_mass'],   # derivatized
                'bioreactions': list(zip(feat['enzyme'], feat['substrate'])),
                'rxn_steps': [_steps],
                'inferred_formula': _steps[-1][1],      # not derivatized
                'synonyms': [_steps[-1][0]]
            }
        else:
            unmatched.append(feat)

print(len(DnCl_matched_features_supernatant), len(unmatched), len(set([f['id'] for f in unmatched])))

In [ ]:
json.dump(list(DnCl_matched_features_supernatant.values()), 
          open(os.path.join(outdir, 'DnCl_anno_features_supernatant.json'), 'w'), indent=4)
json.dump(unmatched, open(os.path.join(outdir, 'DnCl_unmatched_features_supernatant.json'), 'w'), indent=4)

In [ ]:
# Do DnHz, including dhea, progesterone, pregnenolone
#
DnHz_matched_features_pellets, unmatched = {}, []

for feat in json.load(open(input_dir + 'dhea_3step_products_pellets_features.json')):
    if 'matched' in feat:
        _steps = summarize_rxn_steps_per_feature(feat['matched'])
        DnHz_matched_features_pellets[feat['id']] = {
            'id': feat['id'],
            'mz': feat['mz'],
            'rtime': feat['rtime'],
            'source': feat['source'],
            "ion_relation": feat['ion_relation'],
            "neutral_mass": feat['neutral_formula_mass'],   # derivatized
            'bioreactions': list(zip(feat['enzyme'], feat['substrate'])),
            'rxn_steps': [_steps],
            'inferred_formula': _steps[-1][1],      # not derivatized
            'synonyms': [_steps[-1][0]]
        }

for feat in json.load(open(input_dir + 'progesterone_3step_products_pellets_features.json')):
    if 'matched' in feat:
        _steps = summarize_rxn_steps_per_feature(feat['matched'])
        if feat['id'] in DnHz_matched_features_pellets:
            DnHz_matched_features_pellets[feat['id']]['rxn_steps'].append(_steps)
            DnHz_matched_features_pellets[feat['id']]['synonyms'].append(_steps[-1][0])
        else:
            DnHz_matched_features_pellets[feat['id']] = {
                'id': feat['id'],
                'mz': feat['mz'],
                'rtime': feat['rtime'],
                'source': feat['source'],
                "ion_relation": feat['ion_relation'],
                "neutral_mass": feat['neutral_formula_mass'],   # derivatized
                'bioreactions': list(zip(feat['enzyme'], feat['substrate'])),
                'rxn_steps': [_steps],
                'inferred_formula': _steps[-1][1],      # not derivatized
                'synonyms': [_steps[-1][0]]
            }

for feat in json.load(open(input_dir + 'pregnenolone_3step_products_pellets_features.json')):
    if 'matched' in feat:
        _steps = summarize_rxn_steps_per_feature(feat['matched'])
        if feat['id'] in DnHz_matched_features_pellets:
            DnHz_matched_features_pellets[feat['id']]['rxn_steps'].append(_steps)
            DnHz_matched_features_pellets[feat['id']]['synonyms'].append(_steps[-1][0])
        else:
            DnHz_matched_features_pellets[feat['id']] = {
                'id': feat['id'],
                'mz': feat['mz'],
                'rtime': feat['rtime'],
                'source': feat['source'],
                "ion_relation": feat['ion_relation'],
                "neutral_mass": feat['neutral_formula_mass'],   # derivatized
                'bioreactions': list(zip(feat['enzyme'], feat['substrate'])),
                'rxn_steps': [_steps],
                'inferred_formula': _steps[-1][1],      # not derivatized
                'synonyms': [_steps[-1][0]]
            }

for feat in json.load(open(input_dir + 'dhea_3step_products_pellets_unmatched.json')
                      ) + json.load(open(input_dir + 'progesterone_3step_products_pellets_unmatched.json')
                                    ) + json.load(open(input_dir + 'pregnenolone_3step_products_pellets_unmatched.json')):
    if feat['id'] not in DnHz_matched_features_pellets:
        if 'matched' in feat and feat['id'] not in DnHz_matched_features_pellets:
            _steps = summarize_rxn_steps_per_feature(feat['matched'])
            DnHz_matched_features_pellets[feat['id']] = {
                'id': feat['id'],
                'mz': feat['mz'],
                'rtime': feat['rtime'],
                'source': feat['source'],
                "ion_relation": feat['ion_relation'],
                "neutral_mass": feat['neutral_formula_mass'],   # derivatized
                'bioreactions': list(zip(feat['enzyme'], feat['substrate'])),
                'rxn_steps': [_steps],
                'inferred_formula': _steps[-1][1],      # not derivatized
                'synonyms': [_steps[-1][0]]
            }
        else:
            unmatched.append(feat)
            
print(len(DnHz_matched_features_pellets), len(unmatched), len(set([f['id'] for f in unmatched])))

In [ ]:
json.dump(list(DnHz_matched_features_pellets.values()), 
          open(os.path.join(outdir, 'DnHz_anno_features_pellets.json'), 'w'), indent=4)
json.dump(unmatched, open(os.path.join(outdir, 'DnHz_unmatched_features_pellets.json'), 'w'), indent=4)

In [ ]:
DnHz_matched_features_supernatant, unmatched = {}, []

for feat in json.load(open(input_dir + 'dhea_3step_products_supernatant_features.json')):
    if 'matched' in feat:
        _steps = summarize_rxn_steps_per_feature(feat['matched'])
        DnHz_matched_features_supernatant[feat['id']] = {
            'id': feat['id'],
            'mz': feat['mz'],
            'rtime': feat['rtime'],
            'source': feat['source'],
            "ion_relation": feat['ion_relation'],
            "neutral_mass": feat['neutral_formula_mass'],   # derivatized
            'bioreactions': list(zip(feat['enzyme'], feat['substrate'])),
            'rxn_steps': [_steps],
            'inferred_formula': _steps[-1][1],      # not derivatized
            'synonyms': [_steps[-1][0]]
        }

for feat in json.load(open(input_dir + 'progesterone_3step_products_supernatant_features.json')):
    if 'matched' in feat:
        _steps = summarize_rxn_steps_per_feature(feat['matched'])
        if feat['id'] in DnHz_matched_features_supernatant:
            DnHz_matched_features_supernatant[feat['id']]['rxn_steps'].append(_steps)
            DnHz_matched_features_supernatant[feat['id']]['synonyms'].append(_steps[-1][0])
        else:
            DnHz_matched_features_supernatant[feat['id']] = {
                'id': feat['id'],
                'mz': feat['mz'],
                'rtime': feat['rtime'],
                'source': feat['source'],
                "ion_relation": feat['ion_relation'],
                "neutral_mass": feat['neutral_formula_mass'],   # derivatized
                'bioreactions': list(zip(feat['enzyme'], feat['substrate'])),
                'rxn_steps': [_steps],
                'inferred_formula': _steps[-1][1],      # not derivatized
                'synonyms': [_steps[-1][0]]
            }

for feat in json.load(open(input_dir + 'pregnenolone_3step_products_supernatant_features.json')):
    if 'matched' in feat:
        _steps = summarize_rxn_steps_per_feature(feat['matched'])
        if feat['id'] in DnHz_matched_features_supernatant:
            DnHz_matched_features_supernatant[feat['id']]['rxn_steps'].append(_steps)
            DnHz_matched_features_supernatant[feat['id']]['synonyms'].append(_steps[-1][0])
        else:
            DnHz_matched_features_supernatant[feat['id']] = {
                'id': feat['id'],
                'mz': feat['mz'],
                'rtime': feat['rtime'],
                'source': feat['source'],
                "ion_relation": feat['ion_relation'],
                "neutral_mass": feat['neutral_formula_mass'],   # derivatized
                'bioreactions': list(zip(feat['enzyme'], feat['substrate'])),
                'rxn_steps': [_steps],
                'inferred_formula': _steps[-1][1],      # not derivatized
                'synonyms': [_steps[-1][0]]
            }

for feat in json.load(open(input_dir + 'dhea_3step_products_supernatant_unmatched.json')
                      ) + json.load(open(input_dir + 'progesterone_3step_products_supernatant_unmatched.json')
                                    ) + json.load(open(input_dir + 'pregnenolone_3step_products_supernatant_unmatched.json')):
    if feat['id'] not in DnHz_matched_features_supernatant:
        if 'matched' in feat and feat['id'] not in DnHz_matched_features_supernatant:
            _steps = summarize_rxn_steps_per_feature(feat['matched'])
            DnHz_matched_features_supernatant[feat['id']] = {
                'id': feat['id'],
                'mz': feat['mz'],
                'rtime': feat['rtime'],
                'source': feat['source'],
                "ion_relation": feat['ion_relation'],
                "neutral_mass": feat['neutral_formula_mass'],   # derivatized
                'bioreactions': list(zip(feat['enzyme'], feat['substrate'])),
                'rxn_steps': [_steps],
                'inferred_formula': _steps[-1][1],      # not derivatized
                'synonyms': [_steps[-1][0]]
            }
        else:
            unmatched.append(feat)
            
print(len(DnHz_matched_features_supernatant), len(unmatched), len(set([f['id'] for f in unmatched])))


In [ ]:
json.dump(list(DnHz_matched_features_supernatant.values()), 
          open(os.path.join(outdir, 'DnHz_anno_features_supernatant.json'), 'w'), indent=4)
json.dump(unmatched, open(os.path.join(outdir, 'DnHz_unmatched_features_supernatant.json'), 'w'), indent=4)

### Write tsv from JSON

- feature annotation
- build CytoScape network files

In [ ]:
# how many overlap btw cells & supernatant, 10 ppm (almost same 5 ppm)
_m = mf.list_match_lcms_features(list(DnCl_matched_features_pellets.values()), 
                                 list(DnCl_matched_features_supernatant.values()), 10, 2)

In [ ]:
_m['F10234']

In [ ]:
285 + 47 - 21 + 107 + 276 - 13  # rxn matched total features

In [ ]:
merged_DnCl_features = []
overlap = []
for L in _m.values(): overlap += L

for feat in DnCl_matched_features_pellets.values():
    feat['feature_id_pellets'] = feat['id']
    feat['bioreactions_pellets'] = feat['bioreactions']
    matched_ids = _m.get(feat['id'], [])
    feat['feature_id_supernatant'] = ';'.join(matched_ids) if matched_ids else 'NA'
    feat['bioreactions_supernatant'] = [DnCl_matched_features_supernatant[i]['bioreactions'] for i in matched_ids] if matched_ids else 'NA'
    feat.pop('bioreactions')
    if feat['id'] in _m:
        feat['source'] += ';supernatant'
    merged_DnCl_features.append(feat)
    
for feat in DnCl_matched_features_supernatant.values():
    if feat['id'] not in overlap:
        feat['feature_id_pellets'] = 'NA'
        feat['bioreactions_pellets'] = 'NA'
        feat['feature_id_supernatant'] = feat['id']
        feat['bioreactions_supernatant'] = [feat['bioreactions']]
        feat.pop('bioreactions')
        merged_DnCl_features.append(feat)
        
print(len(merged_DnCl_features), 
      len(DnCl_matched_features_pellets), len(DnCl_matched_features_supernatant)
      )

In [ ]:
# redundant features but more within a file
unmatched__DnCl_features = []
tracker = []
for f in json.load(open(os.path.join(outdir, 'DnCl_unmatched_features_pellets.json'))
                   ) + json.load(open(os.path.join(outdir, 'DnCl_unmatched_features_supernatant.json'))):
    _id_ = str(f['mz']) + '@' + str(f['rtime'])
    if _id_ not in tracker:
        tracker.append(_id_)
        unmatched__DnCl_features.append(f)
   
print(len(unmatched__DnCl_features))

In [ ]:
merged_DnCl_features[0].keys()

In [ ]:
# Export tsv
header = ['id', 'mz', 'rtime', 'source', 'ion_relation', 'neutral_mass', 
          'bioreactions_pellets', 'bioreactions_supernatant',
          'rxn_steps', 'inferred_formula', 'synonyms',
          'feature_id_pellets', 'feature_id_supernatant']
s = '\t'.join(header) + '\n'
for feat in merged_DnCl_features:
    row = '\t'.join([str(feat.get(h, '')) for h in header]) + '\n'
    s += row
    
for feat in unmatched__DnCl_features:
    bioreactions = list(zip(feat['enzyme'], feat['substrate']))
    is_pellets = feat['source'].endswith('_pellets')
    feat['bioreactions_pellets'] = bioreactions if is_pellets else 'NA'
    feat['bioreactions_supernatant'] = 'NA' if is_pellets else [bioreactions]
    feat['feature_id_pellets'] = feat['id'] if is_pellets else 'NA'
    feat['feature_id_supernatant'] = 'NA' if is_pellets else feat['id']
    row = '\t'.join([str(feat.get(h, '')) for h in header]) + '\n'
    s += row
    
with open(os.path.join(outdir, 'rxn_annotated_DnCl_features.tsv'), 'w') as f:
    f.write(s)

In [ ]:
_m2 = mf.list_match_lcms_features(list(DnHz_matched_features_pellets.values()), 
                                  list(DnHz_matched_features_supernatant.values()), 10, 2) 

In [ ]:
merged_DnHz_features = []
overlap = []
for L in _m2.values(): overlap += L

for feat in DnHz_matched_features_pellets.values():
    feat['feature_id_pellets'] = feat['id']
    feat['bioreactions_pellets'] = feat['bioreactions']
    matched_ids = _m2.get(feat['id'], [])
    feat['feature_id_supernatant'] = ';'.join(matched_ids) if matched_ids else 'NA'
    feat['bioreactions_supernatant'] = [DnHz_matched_features_supernatant[i]['bioreactions'] for i in matched_ids] if matched_ids else 'NA'
    feat.pop('bioreactions')
    if feat['id'] in _m2:
        feat['source'] += ';supernatant'
    merged_DnHz_features.append(feat)
    
for feat in DnHz_matched_features_supernatant.values():
    if feat['id'] not in overlap:
        feat['feature_id_pellets'] = 'NA'
        feat['bioreactions_pellets'] = 'NA'
        feat['feature_id_supernatant'] = feat['id']
        feat['bioreactions_supernatant'] = [feat['bioreactions']]
        feat.pop('bioreactions')
        merged_DnHz_features.append(feat)
        
print(len(merged_DnHz_features), 
      len(DnHz_matched_features_pellets), len(DnHz_matched_features_supernatant)
      )

In [ ]:
unmatched__DnHz_features = []
tracker = []
for f in json.load(open(os.path.join(outdir, 'DnHz_unmatched_features_pellets.json'))
                   ) + json.load(open(os.path.join(outdir, 'DnHz_unmatched_features_supernatant.json'))):
    _id_ = str(f['mz']) + '@' + str(f['rtime'])
    if _id_ not in tracker:
        tracker.append(_id_)
        unmatched__DnHz_features.append(f)
   
print(len(unmatched__DnHz_features))

In [ ]:
# Export tsv
header = ['id', 'mz', 'rtime', 'source', 'ion_relation', 'neutral_mass', 
          'bioreactions_pellets', 'bioreactions_supernatant',
          'rxn_steps', 'inferred_formula', 'synonyms',
          'feature_id_pellets', 'feature_id_supernatant']
s = '\t'.join(header) + '\n'
for feat in merged_DnHz_features:
    row = '\t'.join([str(feat.get(h, '')) for h in header]) + '\n'
    s += row
    
for feat in unmatched__DnHz_features:
    bioreactions = list(zip(feat['enzyme'], feat['substrate']))
    is_pellets = feat['source'].endswith('_pellets')
    feat['bioreactions_pellets'] = bioreactions if is_pellets else 'NA'
    feat['bioreactions_supernatant'] = 'NA' if is_pellets else [bioreactions]
    feat['feature_id_pellets'] = feat['id'] if is_pellets else 'NA'
    feat['feature_id_supernatant'] = 'NA' if is_pellets else feat['id']
    row = '\t'.join([str(feat.get(h, '')) for h in header]) + '\n'
    s += row
    
with open(os.path.join(outdir, 'rxn_annotated_DnHz_features.tsv'), 'w') as f:
    f.write(s)

In [ ]:
json.dump(merged_DnCl_features, open(os.path.join(outdir, 'merged_DnCl_features.json'), 'w'), indent=2)
json.dump(merged_DnHz_features, open(os.path.join(outdir, 'merged_DnHz_features.json'), 'w'), indent=2)

## Summary (I)

Written rxn annotated features. 

Feature IDs were from both pellet and supernant (pellet when redundant), thus possibly not unique (just F12606). 
Combining with source is unique.

These two tables will be used to compare to KEGG and human GEM later.

In [ ]:
# is F id unique?
len(merged_DnHz_features), len(set([f['id'] for f in merged_DnHz_features])), len(
    merged_DnCl_features), len(set([f['id'] for f in merged_DnCl_features]))

In [ ]:
x = [f['id'] for f in merged_DnCl_features]
[ii for ii in set(x) if x.count(ii) > 1]

## Port network files

Only use ['M0,M+H+'] features. 

In [ ]:
substrates = {
    'Estradiol': {
        'mz': 506.23596,        # DnCl
            'formula': 'C18H24O2',
            'formula_dict': {'C': 18, 'H': 24, 'O': 2},
            'rtime': 1094, 
            'id': 'Estradiol'
    },
    'Estrone': {
        'mz': 504.22031,
        'formula': 'C18H22O2',
        'formula_dict': {'C': 18, 'H': 22, 'O': 2},
        'rtime': 1099,
        'id': 'Estrone'
    },
    'DHEA': {
        'mz': 536.29414,        # DnHz
        'formula': 'C19H28O2',
        'formula_dict': {'C': 19, 'H': 28, 'O': 2},
        'rtime': 549,
        'id': 'DHEA'
    },
    'Pregnenolone': {
        'mz': 564.32544,
        'formula': 'C21H32O2',
        'formula_dict': {'C': 21, 'H': 32, 'O': 2},
        'rtime': 640,
        'id': 'Pregnenolone'
    },
    'Progesterone': {
        'mz': 562.30979,
        'formula': 'C21H30O2',
        'formula_dict': {'C': 21, 'H': 30, 'O': 2},
        'rtime': 695,
        'id': 'Progesterone'
    }
}

In [ ]:
nodeList = []
synonymDict = {}
for feat in merged_DnCl_features + merged_DnHz_features:
    if 'M0,M+H+' in feat['ion_relation']:
        feat['uid'] = feat['id'] + '_' + feat['source']
        nodeList.append(feat)
        for synonym in feat['synonyms']:
            synonymDict[synonym] = feat['uid']
            
for k, sub in substrates.items():
    nodeList.append({
        'uid': sub['id'],
        'id': sub['id'],
        'mz': sub['mz'],
        'rtime': sub['rtime'],
        'source': 'substrate',
        'ion_relation': 'M0,M+H+',
        'inferred_formula': sub['formula'],
        'bioreactions_pellets': 'NA',
        'bioreactions_supernatant': 'NA'
    })
            
header = ['uid', 'id', 'mz', 'rtime', 'source', 'ion_relation', 'inferred_formula',
          'bioreactions_pellets', 'bioreactions_supernatant']
s = '\t'.join(header) + '\n'
for row in nodeList:
    s += '\t'.join([str(row.get(h, '')) for h in header]) + '\n'
    
with open(os.path.join(outdir, 'nodes_merged_metabolites.txt'), 'w') as f:
    f.write(s)

In [ ]:
edgeList = []
for feat in merged_DnCl_features + merged_DnHz_features:
    if 'M0,M+H+' in feat['ion_relation']:
        for rxn_steps in feat['rxn_steps']: # nested lists
            for ii in range(len(rxn_steps) - 1):
                # source, target, rxn formula delta, rxn name
                edgeList.append((rxn_steps[ii][0], rxn_steps[ii + 1][0], rxn_steps[ii][2], rxn_steps[ii][3]))

header = ['source', 'target', 'rxn_formula_delta', 'rxn_name']
s = '\t'.join(header) + '\n'
for row in set(edgeList):
    # replace synonyms
    new = (synonymDict.get(row[0], row[0]), synonymDict.get(row[1], row[1]), row[2], row[3])
    s += '\t'.join([str(x) for x in new]) + '\n'
    
with open(os.path.join(outdir, 'edges_merged_metabolites.txt'), 'w') as f:
    f.write(s)

In [ ]:
len(edgeList), len(set(edgeList))

## Summary (II)

Exported combined overall feature-level network files. 

Next compare KEGG formulas and ours.

In [ ]:
dncl_formulas = [f['inferred_formula'] for f in merged_DnCl_features if 'inferred_formula' in f]
len(dncl_formulas), len(set(dncl_formulas))

In [ ]:
dnhz_formulas = [f['inferred_formula'] for f in merged_DnHz_features if 'inferred_formula' in f]
len(dnhz_formulas), len(set(dnhz_formulas))

In [ ]:
kegg_formulas =  pd.read_csv('input/KEGG/map00140_cytoscape_nodes.csv').to_dict(orient='records')
len(kegg_formulas)

In [ ]:
kegg_set = set([x['formula'] for x in kegg_formulas])
len(kegg_set), len(kegg_set.intersection(set(dncl_formulas))), len(kegg_set.intersection(set(dnhz_formulas)))


In [ ]:
kegg_set.intersection(set(dncl_formulas))

In [ ]:
all_our_formulas = set(dncl_formulas + dnhz_formulas + [x['formula'] for x in substrates.values()])
len(all_our_formulas), len(kegg_set.intersection(all_our_formulas)), len(kegg_set)
#
# This is overlap at formula level with KEGG
#

In [ ]:
# write formula node attributes
overlap_formulas = kegg_set.intersection(all_our_formulas)
s = 'formula\tpresence\n'
for f in overlap_formulas:
    s += f + '\tBoth\n'
for f in all_our_formulas:
    if f not in overlap_formulas:
        s += f + '\tOur\n'
for f in kegg_set:
    if f not in all_our_formulas:
        s += f + '\tKEGG\n'
with open(os.path.join(outdir, 'formula_attributes.txt'), 'w') as f:
    f.write(s)

## Summary

This notebook combined supernatant and cell data, and built reaction networks. 

They serve as annotation for steroid features found in this work. 

Additional network files are generated in this step, for overall feature network view. 

An overlap attribute file is generated on overlap btw KEGG and ours at formula level.